# TP — Méthodes d'Euler et transport numérique

**Analyse numérique — ESTP PGE1 S6 — 2025-2026**

---

## Consignes

- Ce notebook est **le sujet et le rendu**. Complétez les cellules de code marquées `# À COMPLÉTER` et répondez aux questions dans les cellules prévues.
- Exécutez chaque cellule dans l'ordre (**Maj+Entrée**).
- Le notebook doit **s'exécuter sans erreur** de bout en bout.
- Les figures doivent comporter des **titres, axes et légendes** lisibles.
- Les réponses aux questions doivent être **argumentées** (2-3 phrases minimum).

## Barème

| Partie | Contenu | Points |
|--------|---------|--------|
| 1 | Euler explicite sur le café | /7 |
| 2 | Transport d'un colorant (schéma décentré amont) | /8 |
| 3 | Instabilité et condition CFL | /5 |
| Bonus | Pendule animé ou schéma implicite | +2 |
| **Total** | | **/20 (+2)** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 12})

---

# Partie 1 — Euler explicite sur le café (/7 pts)

Au CM1, on a modélisé le refroidissement d'un café par l'EDO :

$$\frac{dT}{dt} = -k\,(T - T_{\text{amb}}), \qquad T(0) = T_0$$

avec $k = 0{,}1\;\text{min}^{-1}$, $T_0 = 90\;°\text{C}$, $T_{\text{amb}} = 20\;°\text{C}$.

La méthode d'Euler explicite approche la solution pas à pas :

$$y_{n+1} = y_n + \Delta t \, f(t_n,\, y_n)$$

## Exercice 1.1 — Implémenter Euler explicite (2 pts)

Compléter la fonction ci-dessous. Elle prend en entrée :
- `f` : la fonction $f(t, y)$ de l'EDO $y' = f(t, y)$
- `y0` : la condition initiale
- `t0`, `tf` : les bornes de l'intervalle de temps
- `dt` : le pas de temps

Elle renvoie deux tableaux NumPy : les instants $t_0, t_1, \ldots$ et les valeurs approchées $y_0, y_1, \ldots$

**Rappel :** $y_{n+1} = y_n + \Delta t \cdot f(t_n,\, y_n)$.

In [ ]:
def euler_explicite(f, y0, t0, tf, dt):
    """Méthode d'Euler explicite pour y' = f(t, y)."""
    n = int((tf - t0) / dt)
    t = np.zeros(n + 1)
    y = np.zeros(n + 1)
    t[0] = t0
    y[0] = y0

    for i in range(n):
        t[i + 1] = t[i] + dt
        y[i + 1] = ...  # À COMPLÉTER

    return t, y

In [ ]:
# --- Vérification automatique ---
# EDO test : y' = -y, y(0) = 1  =>  y(t) = exp(-t)
t_test, y_test = euler_explicite(lambda t, y: -y, 1.0, 0.0, 1.0, 0.01)
erreur_test = np.max(np.abs(y_test - np.exp(-t_test)))

print(f"Erreur max sur le test : {erreur_test:.6f}")
assert erreur_test < 0.01, "L'erreur semble trop grande, vérifiez votre formule."
print("✓ Test réussi !")

## Exercice 1.2 — Application au café (1 pt)

Définir la fonction `f_cafe(t, T)` correspondant à l'EDO du café, puis appeler `euler_explicite` avec $\Delta t = 2$ min sur l'intervalle $[0,\, 30]$ min.

**Rappel :** $f(t, T) = -k\,(T - T_{\text{amb}})$ avec $k = 0{,}1$ et $T_{\text{amb}} = 20$.

In [ ]:
k = 0.1
T0 = 90.0
T_amb = 20.0
dt = 2.0
tf = 30.0

def f_cafe(t, T):
    return ...  # À COMPLÉTER

t_euler, T_euler = euler_explicite(f_cafe, T0, 0.0, tf, dt)

In [ ]:
# --- Tracé fourni ---
t_exact = np.linspace(0, tf, 300)
T_exact = T_amb + (T0 - T_amb) * np.exp(-k * t_exact)

plt.figure()
plt.plot(t_exact, T_exact, "k-", linewidth=2, label="Solution exacte")
plt.plot(t_euler, T_euler, "o--", color="C0", label=f"Euler (Δt = {dt} min)")
plt.xlabel("Temps (min)")
plt.ylabel("Température (°C)")
plt.title("Refroidissement du café — Euler vs exact")
plt.legend()
plt.grid(True)
plt.show()

## Exercice 1.3 — Influence du pas de temps (2 pts)

Le code ci-dessous trace la solution d'Euler pour plusieurs pas de temps. **Exécuter**, puis répondre : quel pas donne le résultat le plus proche de la solution exacte ? Que gagne-t-on en diminuant le pas ? Que perd-on ?

In [ ]:
pas_a_tester = [5, 2, 1, 0.5]

plt.figure()
plt.plot(t_exact, T_exact, "k-", linewidth=2, label="Exacte")

for dt_test in pas_a_tester:
    t_e, T_e = euler_explicite(f_cafe, T0, 0.0, tf, dt_test)
    plt.plot(t_e, T_e, "o--", markersize=4, label=f"Δt = {dt_test} min")

plt.xlabel("Temps (min)")
plt.ylabel("Température (°C)")
plt.title("Influence du pas de temps")
plt.legend()
plt.grid(True)
plt.show()

**Votre réponse :** *(double-cliquez pour éditer)*



## Exercice 1.4 — Erreur en fonction du pas (1 pt)

Pour chaque pas $\Delta t$ dans la liste ci-dessous, calculer l'erreur maximale :

$$\max_i \left|T_{\text{exact}}(t_i) - T_i^{\text{Euler}}\right|$$

et la stocker dans la liste `erreurs`.

In [ ]:
pas_a_tester = [5, 2, 1, 0.5, 0.2, 0.1]
erreurs = []

for dt_test in pas_a_tester:
    t_e, T_e = euler_explicite(f_cafe, T0, 0.0, tf, dt_test)
    T_ex = T_amb + (T0 - T_amb) * np.exp(-k * t_e)
    err = ...  # À COMPLÉTER : erreur maximale entre T_e et T_ex
    erreurs.append(err)

# --- Tracé fourni ---
plt.figure()
plt.loglog(pas_a_tester, erreurs, "o-", color="C1", linewidth=2)
plt.xlabel("Pas de temps Δt (min)")
plt.ylabel("Erreur maximale (°C)")
plt.title("Convergence de la méthode d'Euler")
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()

## Question 1.5 — Que se passe-t-il si le pas est trop grand ? (1 pt)

Exécuter le code ci-dessous, puis répondre : le résultat obtenu avec $\Delta t = 15$ min est-il physiquement crédible ? Pourquoi ?

In [ ]:
t_big, T_big = euler_explicite(f_cafe, T0, 0.0, tf, 15.0)

plt.figure()
plt.plot(t_exact, T_exact, "k-", linewidth=2, label="Exacte")
plt.plot(t_big, T_big, "s-", color="red", linewidth=2, label="Euler (Δt = 15 min)")
plt.axhline(y=0, color="gray", linestyle=":", alpha=0.5)
plt.xlabel("Temps (min)")
plt.ylabel("Température (°C)")
plt.title("Euler avec un pas trop grand")
plt.legend()
plt.grid(True)
plt.show()

print(f"T après 15 min : {T_big[1]:.1f} °C")

**Votre réponse :** *(double-cliquez pour éditer)*



---

# Partie 2 — Transport d'un colorant (/8 pts)

Au CM2, on a modélisé le transport d'un colorant dans un canal par l'EDP :

$$\frac{\partial u}{\partial t} + c\,\frac{\partial u}{\partial x} = 0$$

avec $c = 2\;\text{m/s}$ (vitesse du courant). La solution exacte est une **translation pure** : $u(x,t) = u_0(x - ct)$.

En combinant Euler (temps) et différences finies rétrogrades (espace), on a obtenu le **schéma décentré amont** :

$$u_i^{n+1} = u_i^n - r\,(u_i^n - u_{i-1}^n), \qquad r = \frac{c\,\Delta t}{\Delta x}$$

## Exercice 2.1 — Implémenter le schéma décentré amont (2 pts)

Compléter la fonction ci-dessous. Elle renvoie un tableau 2D `U` de taille `(n_steps+1, M)` où `M` est le nombre de nœuds et `n_steps` le nombre de pas de temps.

**Rappel :** $u_i^{n+1} = u_i^n - r\,(u_i^n - u_{i-1}^n)$ avec $r = c\,\Delta t / \Delta x$.

In [ ]:
def transport_upwind(u0, c, dx, dt, n_steps):
    """Schéma décentré amont (upwind) pour l'équation de transport."""
    M = len(u0)
    r = c * dt / dx
    U = np.zeros((n_steps + 1, M))
    U[0, :] = u0.copy()

    for n in range(n_steps):
        for i in range(1, M):
            U[n + 1, i] = ...  # À COMPLÉTER
        U[n + 1, 0] = U[n, 0]

    return U

In [ ]:
# --- Vérification automatique ---
# Avec r = 1, le créneau se translate exactement d'un noeud par pas de temps
u0_test = np.array([0, 0, 1, 1, 0, 0, 0, 0], dtype=float)
U_test = transport_upwind(u0_test, c=2.0, dx=2.0, dt=1.0, n_steps=1)

attendu = np.array([0, 0, 0, 1, 1, 0, 0, 0], dtype=float)
assert np.allclose(U_test[1], attendu), f"Résultat inattendu : {U_test[1]}"
print("✓ Test réussi (r = 1 : translation exacte d'un noeud) !")

## Exercice 2.2 — Retrouver les résultats du CM2 (2 pts)

Reproduire le calcul fait à la main en cours : canal de 14 m, 8 nœuds ($\Delta x = 2$ m), $\Delta t = 0{,}5$ s, $c = 2$ m/s, soit $r = 0{,}5$.

Condition initiale : concentration $u = 1$ g/L entre $x = 4$ m et $x = 6$ m, nulle ailleurs.

Vérifier que vos résultats correspondent au tableau du poly.

In [ ]:
c = 2.0
dx = 2.0
dt = 0.5

u0_cm2 = np.array([0, 0, 1, 1, 0, 0, 0, 0], dtype=float)
U_cm2 = transport_upwind(u0_cm2, c, dx, dt, n_steps=2)

# --- Affichage ---
print("Résultats (à comparer avec le tableau du CM2) :")
print()
print(f"{'n':>3} {'t (s)':>7}  " + "  ".join([f"  u_{i}" for i in range(8)]))
print("-" * 65)
for step in range(3):
    vals = "  ".join([f"{U_cm2[step, i]:5.2f}" for i in range(8)])
    print(f"{step:>3} {step * dt:>7.1f}  {vals}")

## Exercice 2.3 — Maillage fin et animation (2 pts)

On passe à un maillage fin (200 nœuds) pour mieux visualiser le phénomène. Le code ci-dessous appelle votre fonction `transport_upwind` et génère une **animation** du colorant se déplaçant dans le canal.

Exécuter et observer. L'animation peut prendre quelques secondes à s'afficher.

In [ ]:
L = 20.0
M = 200
c = 2.0
dx = L / (M - 1)
dt = 0.4 * dx / c
n_steps = 300

x = np.linspace(0, L, M)
u0_fin = np.where((x >= 2) & (x <= 4), 1.0, 0.0)

U_upwind = transport_upwind(u0_fin, c, dx, dt, n_steps)

# --- Animation fournie ---
fig, ax = plt.subplots()
line, = ax.plot(x, U_upwind[0, :], "C0", linewidth=2)
ax.set_xlim(0, L)
ax.set_ylim(-0.2, 1.4)
ax.set_xlabel("Position x (m)")
ax.set_ylabel("Concentration u (g/L)")
ax.set_title("Transport — décentré amont")
ax.grid(True)

def _update_upwind(frame):
    line.set_ydata(U_upwind[frame, :])
    ax.set_title(f"Transport — décentré amont — t = {frame * dt:.2f} s")

anim_upwind = FuncAnimation(
    fig, _update_upwind,
    frames=range(0, n_steps + 1, 3), interval=50)
plt.close(fig)
HTML(anim_upwind.to_jshtml())

## Exercice 2.4 — Comparaison avec la solution exacte (1 pt)

La solution exacte est une translation pure : le créneau se déplace à vitesse $c$ sans se déformer. Le code ci-dessous superpose la solution numérique et la solution exacte au temps final.

In [ ]:
t_final = n_steps * dt
u_exact_final = np.where(
    (x >= 2 + c * t_final) & (x <= 4 + c * t_final), 1.0, 0.0)

plt.figure()
plt.plot(x, u_exact_final, "k--", linewidth=2, label="Solution exacte (translation)")
plt.plot(x, U_upwind[-1, :], "C0", linewidth=2, label="Schéma décentré amont")
plt.plot(x, u0_fin, ":", color="gray", alpha=0.5, label="Condition initiale")
plt.xlabel("Position x (m)")
plt.ylabel("Concentration u (g/L)")
plt.title(f"Comparaison à t = {t_final:.1f} s")
plt.legend()
plt.grid(True)
plt.show()

## Question 2.5 — Diffusion numérique (1 pt)

Le profil numérique est-il une translation exacte du créneau initial ? Quel défaut observez-vous ? Comment ce défaut évoluerait-il si l'on prenait un maillage plus fin ?

**Votre réponse :** *(double-cliquez pour éditer)*



---

# Partie 3 — Instabilité et condition CFL (/5 pts)

Au CM2, on a vu qu'en remplaçant la différence rétrograde par la **différence centrée**, le schéma devient :

$$u_i^{n+1} = u_i^n - \frac{r}{2}\,(u_{i+1}^n - u_{i-1}^n)$$

Ce schéma, pourtant d'apparence naturelle, produit des résultats catastrophiques.

## Exercice 3.1 — Implémenter le schéma centré (1 pt)

Compléter la fonction ci-dessous. La seule différence avec `transport_upwind` est la formule de mise à jour.

**Rappel :** $u_i^{n+1} = u_i^n - \dfrac{r}{2}\,(u_{i+1}^n - u_{i-1}^n)$.

In [ ]:
def transport_centre(u0, c, dx, dt, n_steps):
    """Schéma centré explicite pour l'équation de transport."""
    M = len(u0)
    r = c * dt / dx
    U = np.zeros((n_steps + 1, M))
    U[0, :] = u0.copy()

    for n in range(n_steps):
        for i in range(1, M - 1):
            U[n + 1, i] = ...  # À COMPLÉTER
        U[n + 1, 0] = U[n, 0]
        U[n + 1, -1] = U[n, -1]

    return U

## Exercice 3.2 — Observer l'instabilité (2 pts)

Le code ci-dessous anime **côte à côte** le schéma décentré amont et le schéma centré, sur les mêmes données que la partie 2. Exécuter et observer attentivement.

In [ ]:
n_compare = 60
U_upwind_short = transport_upwind(u0_fin, c, dx, dt, n_compare)
U_centre_short = transport_centre(u0_fin, c, dx, dt, n_compare)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

line1, = axes[0].plot(x, U_upwind_short[0, :], "C0", linewidth=2)
axes[0].set_xlim(0, L)
axes[0].set_ylim(-3, 4)
axes[0].set_xlabel("x (m)")
axes[0].set_ylabel("u (g/L)")
axes[0].set_title("Décentré amont")
axes[0].grid(True)
axes[0].axhline(y=0, color="gray", linestyle=":", alpha=0.4)

line2, = axes[1].plot(x, U_centre_short[0, :], "C3", linewidth=2)
axes[1].set_xlim(0, L)
axes[1].set_xlabel("x (m)")
axes[1].set_title("Centré")
axes[1].grid(True)
axes[1].axhline(y=0, color="gray", linestyle=":", alpha=0.4)

def _update_compare(frame):
    line1.set_ydata(U_upwind_short[frame, :])
    axes[0].set_title(f"Décentré amont — t = {frame * dt:.2f} s")
    line2.set_ydata(U_centre_short[frame, :])
    axes[1].set_title(f"Centré — t = {frame * dt:.2f} s")

anim_compare = FuncAnimation(
    fig, _update_compare,
    frames=range(0, n_compare + 1, 2), interval=150)
plt.close(fig)
HTML(anim_compare.to_jshtml())

## Question 3.3 — Analyse de l'instabilité (1 pt)

Comparez les deux animations. Qu'observez-vous pour le schéma centré (valeurs négatives, oscillations) ? Pourquoi un schéma qui utilise une approximation « plus symétrique » de la dérivée produit-il un résultat pire que le schéma décentré ?

**Votre réponse :** *(double-cliquez pour éditer)*



## Exercice 3.4 — Condition CFL (1 pt)

Même le schéma décentré amont peut devenir instable si le rapport $r = c\,\Delta t / \Delta x$ est trop grand. Le code ci-dessous teste trois valeurs de $r$. Exécuter et identifier la valeur critique.

In [ ]:
valeurs_r = [0.5, 1.0, 1.5]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, r_val in enumerate(valeurs_r):
    dt_cfl = r_val * dx / c
    n_cfl = int(3.0 / dt_cfl)
    U_cfl = transport_upwind(u0_fin, c, dx, dt_cfl, n_cfl)

    axes[idx].plot(x, u0_fin, "k--", alpha=0.4, label="t = 0")
    axes[idx].plot(
        x, U_cfl[-1, :], "C0", linewidth=2,
        label=f"t = {n_cfl * dt_cfl:.1f} s")
    axes[idx].set_xlim(0, L)
    axes[idx].set_ylim(-1.5, 2.5)
    axes[idx].set_xlabel("x (m)")
    axes[idx].set_title(f"r = {r_val}")
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True)

plt.suptitle("Schéma décentré amont — influence de r", fontsize=14)
plt.tight_layout()
plt.show()

## Question 3.5 — Interpréter la condition CFL (1 pt)

Pour le schéma décentré amont, quelle condition doit vérifier $r$ pour que le schéma reste stable ? Que se passe-t-il concrètement si l'on raffine l'espace ($\Delta x$ plus petit) sans adapter $\Delta t$ ?

**Votre réponse :** *(double-cliquez pour éditer)*



---

# Bonus (+2 pts)

Choisir **une** des deux options ci-dessous et compléter le code correspondant.

## Option A — Pendule animé

Utiliser Euler pour simuler un pendule simple :

$$\begin{cases} \theta'(t) = v(t) \\ v'(t) = -\omega_0^2 \sin(\theta(t)) \end{cases}$$

avec $\omega_0 = 1{,}5\;\text{rad/s}$, $\theta(0) = 0{,}8\;\text{rad}$, $v(0) = 0$.

**À faire :**
1. Compléter la boucle d'Euler pour le système de deux équations.
2. Tracer $\theta(t)$ et le portrait de phase $(\theta, v)$.
3. *(Facultatif)* Ajouter un frottement $-c_f \cdot v$ avec $c_f = 0{,}3$ et comparer.

In [ ]:
omega0 = 1.5
h_pend = 0.02
n_pendule = 2000

t_pend = np.zeros(n_pendule + 1)
theta = np.zeros(n_pendule + 1)
v_pend = np.zeros(n_pendule + 1)

theta[0] = 0.8
v_pend[0] = 0.0

for i in range(n_pendule):
    t_pend[i + 1] = t_pend[i] + h_pend
    theta[i + 1] = ...   # À COMPLÉTER
    v_pend[i + 1] = ...  # À COMPLÉTER

# --- Tracés ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(t_pend, theta)
ax1.set_xlabel("t (s)")
ax1.set_ylabel("θ (rad)")
ax1.set_title("Angle du pendule")
ax1.grid(True)

ax2.plot(theta, v_pend)
ax2.set_xlabel("θ (rad)")
ax2.set_ylabel("v (rad/s)")
ax2.set_title("Portrait de phase")
ax2.grid(True)

plt.tight_layout()
plt.show()

## Option B — Schéma implicite pour le transport

Le schéma implicite décentré amont s'écrit :

$$(1 + r)\,u_i^{n+1} - r\,u_{i-1}^{n+1} = u_i^n$$

Cela revient à résoudre un système linéaire **bidiagonal** à chaque pas de temps.

**À faire :**
1. Compléter la construction de la matrice et la résolution.
2. Tester avec $r = 1{,}5$ et comparer avec le schéma explicite (qui explose).
3. *(Facultatif)* Le schéma implicite est-il toujours stable ? Tester avec $r = 3$.

In [ ]:
def transport_implicite(u0, c, dx, dt, n_steps):
    """Schéma implicite décentré amont pour l'équation de transport."""
    M = len(u0)
    r = c * dt / dx
    U = np.zeros((n_steps + 1, M))
    U[0, :] = u0.copy()

    A = np.eye(M) * (1 + r) + np.eye(M, k=-1) * (-r)
    A[0, 0] = 1.0
    A[0, 1] = 0.0

    for n in range(n_steps):
        rhs = U[n, :].copy()
        U[n + 1, :] = ...  # À COMPLÉTER : résoudre A @ U[n+1,:] = rhs

    return U

# Test avec r = 1.5
dt_imp = 1.5 * dx / c
n_imp = int(3.0 / dt_imp)
U_imp = transport_implicite(u0_fin, c, dx, dt_imp, n_imp)

plt.figure()
plt.plot(x, u0_fin, "k--", alpha=0.4, label="t = 0")
plt.plot(x, U_imp[-1, :], "C2", linewidth=2, label="Implicite (r = 1.5)")
plt.xlabel("x (m)")
plt.ylabel("u (g/L)")
plt.title("Schéma implicite — r = 1.5")
plt.legend()
plt.grid(True)
plt.show()